In [1]:
import uproot
import numpy as np
import pandas as pd
import awkward as ak
import matplotlib.pyplot as plt
import pickle
import os

ROOT_DATA_DIR = "../mu3e_root_data"
signal_data_file = f"{ROOT_DATA_DIR}/run42_sig.root"
background_data_file = f"{ROOT_DATA_DIR}/run42_bg.root"

HIT_COUNT_CUTOFF = 128


with uproot.open(background_data_file) as file:
    keys = file.keys()
    bg_sensor_positions = file["alignment/sensors"].arrays(library="pd")
    bg_mppc_positions = file["alignment/mppcs"].arrays(library="pd")
    bg_fibre_positions = file["alignment/fibres"].arrays(library="pd")
    bg_event_data = file["mu3e"].arrays(library="pd")
with uproot.open(signal_data_file) as file:
    sig_sensor_positions = file["alignment/sensors"].arrays(library="pd")
    sig_mppc_positions = file["alignment/mppcs"].arrays(library="pd")
    sig_fibre_positions = file["alignment/fibres"].arrays(library="pd")
    sig_event_data = file["mu3e"].arrays(library="pd")

FileNotFoundError: [Errno 2] No such file or directory: '/afs/desy.de/user/a/aulich/mu3e_trigger/notebooks/../mu3e_root_data/run42_bg.root'

In [ ]:
def reorder_nla(nla: np.ndarray, padding_value: int = -1) -> np.ndarray:
    """
    Reorders the NLA array to ensure that non-padded entries are at the beginning.
    Assumes padding is identifiable via `nla[:, :, 0] == padding_value`.
    """
    # Identify valid entries
    if nla.ndim == 3:
        valid_mask = nla[:, :, 0] != padding_value
        B, N, D = nla.shape
        flat_nla = nla.reshape(B * N, D)
        flat_valid_mask = valid_mask.reshape(B * N)

        # Get indices of valid entries
        valid_indices = np.nonzero(flat_valid_mask)[0]

        # Allocate output
        reordered_nla = np.full_like(nla, padding_value)

        counts = valid_mask.sum(axis=1)

        # Fill output using advanced indexing
        row_ids = np.repeat(np.arange(B), counts)
        group_counts = np.bincount(row_ids, minlength=B)

        # Compute start indices for placing data
        start_idx = np.zeros_like(group_counts)
        np.cumsum(group_counts[:-1], out=start_idx[1:])

        # Where to write valid entries in each row
        insert_pos = np.hstack([np.arange(c) for c in group_counts])
        reordered_nla[row_ids, insert_pos] = flat_nla[valid_indices]

    else:
        valid_mask = nla != padding_value
        B, N = nla.shape
        flat_nla = nla.reshape(B * N)
        flat_valid_mask = valid_mask.reshape(B * N)

        # Get indices of valid entries
        valid_indices = np.nonzero(flat_valid_mask)[0]

        # Allocate output
        reordered_nla = np.full_like(nla, padding_value)

        counts = valid_mask.sum(axis=1)
        # Fill output using advanced indexing
        row_ids = np.repeat(np.arange(B), counts)
        group_counts = np.bincount(row_ids, minlength=B)

        # Compute start indices for placing data
        start_idx = np.zeros_like(group_counts)
        np.cumsum(group_counts[:-1], out=start_idx[1:])

        # Where to write valid entries in each row
        insert_pos = np.hstack([np.arange(c) for c in group_counts])
        reordered_nla[row_ids, insert_pos] = flat_nla[valid_indices]

    # Compute number of valid entries per batch
    counts = valid_mask.sum(axis=1)
    # Flatten for easier fancy indexing

    return reordered_nla

In [ ]:
def load_ak_series_to_numpy(
    series: pd.Series, max_cols: int = 256, fill_value: int = -1
) -> np.ndarray:
    """
    Converts an Awkward Array Series to a padded NumPy array (2D).
    Pads each row to `max_cols`, truncates longer rows, and ignores empty arrays.
    """
    # Combine series into one awkward array
    ak_array = ak.Array(series.to_list())  # safe if series contains ak Arrays or lists

    # Filter out arrays of invalid lengths
    lengths = ak.num(ak_array)
    mask = (lengths > 0) & (lengths <= max_cols)
    ak_array = ak_array[mask]

    # Clip longer arrays (optional depending on your needs)
    ak_array = ak.pad_none(ak_array, max_cols, clip=True)

    # Replace None with fill_value
    ak_array_filled = ak.fill_none(ak_array, fill_value)

    # Convert to NumPy
    result = ak.to_numpy(ak_array_filled)
    return result

In [ ]:
def load_event_ak_to_numpy(
    series: list[pd.Series],
    cutoff: int = 256,
    fill_value: int = -1,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Converts pixel and MPPC data from Awkward Array Series to padded NumPy arrays.
    Pads each row to `pixel_hit_cutoff` and `mppc_hit_cutoff`, truncates longer rows,
    and ignores empty arrays.
    """
    ak_arrays = [ak.Array(s.to_list()) for s in series]

    ak_lengths = [ak.num(ak_array) for ak_array in ak_arrays]

    ak_masks = [
        (lengths > 0) & (lengths <= cutoff)
        for lengths in ak_lengths]

    combined_mask = ak_masks[0]
    for mask in ak_masks[1:]:
        combined_mask = mask & combined_mask

    masked_arrays = [ak_array[combined_mask] for ak_array in ak_arrays]

    none_padded_arrays = [
        ak.pad_none(ak_array, cutoff, clip=True)
        for ak_array in masked_arrays]


    filled_arrays = [
        ak.to_numpy(ak.fill_none(ak_array, fill_value))
        for ak_array in none_padded_arrays]


    return filled_arrays

In [ ]:
def convert_mppc_to_location(
    mppc: np.ndarray, col_index : np.ndarray, mppc_positions: pd.DataFrame, padding_value: float = -1
) -> np.ndarray:
    """
    Converts a 1D array of MPPC IDs to their corresponding positions in space.
    The MPPC IDs are expected to be in the 'mppc' column of the mppc_positions DataFrame.
    The function returns a 2D array of positions with shape (N, 3), where N is the number of MPPCs.
    Each row corresponds to the (cx, cy, cz) coordinates of an MPPC.

    Parameters:
    mppc (np.ndarray): 1D array of MPPC IDs.
    mppc_positions (pd.DataFrame): DataFrame containing MPPC positions with columns ['mppc', 'cx', 'cy', 'cz'].

    Returns:
    np.ndarray: 2D array of shape (N, 3) containing the (vx, vy) coordinates of the MPPCs.
    """
    # Validate input
    if not isinstance(mppc, np.ndarray) or mppc.ndim != 2:
        raise ValueError("mppc must be a 2D numpy array.")
    
    if mppc.shape != col_index.shape:
        raise ValueError("mppc and col_index must have the same shape.")

    required_columns = ["mppc", "vx", "vy"]

    if not all(col in mppc_positions.columns for col in required_columns):
        raise ValueError(
            f"mppc_positions DataFrame must contain the following columns: {required_columns}"
        )

    mppc_positions = mppc_positions.set_index("mppc")
    locations = np.full((*mppc.shape, 3), dtype=float, fill_value=padding_value)
    flat_locations = locations.reshape(-1, 3)

    flatted_mppc = mppc.flatten()
    flat_mask = flatted_mppc != padding_value

    mppc_data = mppc_positions.loc[flatted_mppc[flat_mask]]

    flat_col_index = col_index.flatten()[flat_mask]

    vx = mppc_data["vx"].to_numpy()
    vy = mppc_data["vy"].to_numpy()
    vz = mppc_data["vz"].to_numpy()

    col_x = mppc_data["colx"].to_numpy()
    col_y = mppc_data["coly"].to_numpy()
    col_z = mppc_data["colz"].to_numpy()

    x = vx + (flat_col_index + 0.5) * col_x
    y = vy + (flat_col_index + 0.5) * col_y
    z = vz + (flat_col_index + 0.5) * col_z
    

    flat_locations[flat_mask] = np.column_stack((x,y,z))

    return locations

In [ ]:
def convert_pid_to_location(
    pixel_id: np.ndarray,
    sensor_positions: pd.DataFrame,
    padding_value: float = -1,
    sensor_fault_rate=0.0,
) -> np.ndarray:
    if sensor_positions.empty:
        raise ValueError("sensor_positions DataFrame is empty.")

    required_columns = [
        "id",
        "vx",
        "vy",
        "vz",
        "rowx",
        "rowy",
        "rowz",
        "colx",
        "coly",
        "colz",
    ]
    if not all(col in sensor_positions.columns for col in required_columns):
        raise ValueError(f"Missing required columns: {required_columns}")

    # Preprocessing
    sensor_positions = sensor_positions.set_index("id", drop=False)
    valid_mask = pixel_id != padding_value

    # Decode pixel_id to chip, col, row
    hit_chip_id = (pixel_id // 2**16).astype(np.int32)
    hit_col_id = ((pixel_id // 2**8) % 2**8).astype(np.float32)
    hit_row_id = (pixel_id % 2**8).astype(np.float32)

    # Flatten inputs for easier indexing
    flat_valid_mask = valid_mask.flatten()
    flat_chip_id = hit_chip_id.flatten()
    flat_col_id = hit_col_id.flatten()
    flat_row_id = hit_row_id.flatten()

    keep_mask = flat_chip_id // 2**12 == 0

    layer_id = ((flat_chip_id // 2**10) % 4) + 1

    # Apply sensor fault rate
    if sensor_fault_rate > 0:
        # If sensor_fault_rate is applied, we keep the mask for faulty sensors
        # but also ensure that layer 1 and 2 are always kept
        # This is because we only assume layers 3 and 4 to be faulty
        keep_mask = (
            (np.random.rand(len(sensor_data)) >= sensor_fault_rate)
            | (layer_id == 1 | layer_id == 2)
        ) & keep_mask

    flat_valid_mask = flat_valid_mask & keep_mask

    # Filter only valid pixel ids
    valid_chip_ids = flat_chip_id[flat_valid_mask]
    valid_cols = flat_col_id[flat_valid_mask] + 0.5
    valid_rows = flat_row_id[flat_valid_mask] + 0.5

    # Lookup transformation vectors
    try:
        sensor_data = sensor_positions.loc[valid_chip_ids]
    except KeyError:
        raise ValueError("Some chip IDs not found in sensor_positions.")

    # Compute positions
    vx = sensor_data["vx"].values
    vy = sensor_data["vy"].values
    vz = sensor_data["vz"].values
    rowx = sensor_data["rowx"].values
    rowy = sensor_data["rowy"].values
    rowz = sensor_data["rowz"].values
    colx = sensor_data["colx"].values
    coly = sensor_data["coly"].values
    colz = sensor_data["colz"].values

    x = vx + valid_cols * colx + valid_rows * rowx
    y = vy + valid_cols * coly + valid_rows * rowy
    z = vz + valid_cols * colz + valid_rows * rowz

    # Fill output array
    location = np.full((*pixel_id.shape, 3), padding_value, dtype=np.float64)
    flat_location = location.reshape(-1, 3)
    flat_location[flat_valid_mask] = np.stack([x, y, z], axis=1)

    return location


def compute_distance(data: np.ndarray, padding_value=-1) -> np.ndarray:
    distance_data = np.linalg.norm(data, axis=-1)
    distance_data[(data == padding_value).all(axis=-1)] = -1
    return distance_data


def sort_by_feature(
    data: np.ndarray, feature: np.ndarray, padding_value=-1
) -> np.ndarray:
    """
    Vectorized version: Sorts `data` along axis 1 by `feature`, respecting a padding mask.
    """
    if data.shape[0:2] != feature.shape[0:2]:
        raise ValueError(
            "Data and feature must have the same number of rows and columns."
        )

    # Create a valid mask over first 2 dims
    if data.ndim == 3:
        valid_mask = (data != padding_value).any(axis=-1)  # shape (B, N)
    elif data.ndim == 2:
        valid_mask = data != padding_value
    else:
        raise ValueError("Data must be 2D or 3D array.")

    # Number of valid elements per row
    num_valid = valid_mask.sum(axis=1)

    # argsort feature, ignoring invalid entries
    sort_indices = np.argsort(np.where(valid_mask, feature, np.inf), axis=1)

    # Prepare output filled with padding_value
    sorted_data = np.full_like(data, padding_value)

    # Use np.take_along_axis to reorder data
    if data.ndim == 3:
        sorted_full = np.take_along_axis(data, sort_indices[:, :, None], axis=1)
    elif data.ndim == 2:
        sorted_full = np.take_along_axis(data, sort_indices[:, :], axis=1)
    else:
        raise ValueError("Data must be 2D or 3D array.")

    # Mask out only the valid sorted parts
    for i, n_valid in enumerate(num_valid):
        if n_valid > 0:
            sorted_data[i, :n_valid] = sorted_full[i, :n_valid]

    return sorted_data

In [ ]:
def adjust_timestamps(timestamps: np.ndarray, padding_value: int = -1) -> np.ndarray:
    """
    Adjusts timestamps based on the data mask.
    If data is padded, the corresponding timestamps are set to -1.
    """
    masked_timestamps = np.where(timestamps != padding_value, timestamps, np.inf)
    min_timestamps = masked_timestamps.min(axis=1, keepdims=True)
    adjusted_timestamps = np.where(
        timestamps != padding_value, timestamps - min_timestamps, -1
    )
    return adjusted_timestamps

In [ ]:
def get_spacetime_from_file(file_path : str,out_dir : str , out_name : str, padding_value : float = -1, hit_cutoff : int = 128):
    """
    Extracts spacetime data from a ROOT file and saves it to a specified directory.
    Parameters:
    - file_path (str): Path to the ROOT file.
    - out_dir (str): Directory to save the output files.
    - out_name (str): Base name for the output files.
    - padding_value (float): Value used for padding in the output arrays. Defaults to -1.
    - hit_cutoff (int): Maximum number of hits per event. Defaults to 128.
    """
    if not file_path.endswith(".root"):
        raise ValueError("File must be a ROOT file with .root extension.")
    if not out_dir.endswith("/"):
        out_dir += "/"
    if not out_name:
        raise ValueError("Output name must be provided.")

    with uproot.open(file_path) as file:
        sensor_positions = file["alignment/sensors"].arrays(library="pd")
        mppc_positions = file["alignment/mppcs"].arrays(library="pd")
        fibre_positions = file["alignment/fibres"].arrays(library="pd")
        event_data = file["mu3e"].arrays(library="pd")
        if sensor_positions.empty or mppc_positions.empty or fibre_positions.empty:
            raise ValueError("Sensor, MPPC, or fibre positions are empty.")
        if event_data.empty:
            raise ValueError("Event data is empty.")
    
    # Convert pixel and MPPC data to numpy arrays
    pixel_hit_id, pixel_hit_timestamp, mppc_id, mppc_col_id, mppc_time  = load_event_ak_to_numpy(
        [event_data["hit_pixelid"],
        event_data["hit_timestamp"],
        event_data["fibremppc_mppc"],
        event_data["fibremppc_col"],
        event_data["fibremppc_time"]],
        cutoff=hit_cutoff,
        fill_value=padding_value,
    )
    if pixel_hit_id.shape != pixel_hit_timestamp.shape:
        raise ValueError("Pixel hit ID and timestamp shapes do not match.")
    if mppc_id.shape != mppc_col_id.shape or mppc_id.shape != mppc_time.shape:
        raise ValueError("MPPC ID, column ID, and time shapes do not match.")
    if pixel_hit_id.shape[0] != mppc_id.shape[0]:
        raise ValueError("Number of events in pixel and MPPC data do not match.")

    # Convert pixel IDs to positions
    pixel_positions = convert_pid_to_location(
        pixel_hit_id,
        sensor_positions,
        padding_value=padding_value,
        sensor_fault_rate=0.0,
    )
    if pixel_positions.shape[0] != pixel_hit_id.shape[0]:
        raise ValueError("Pixel positions shape does not match pixel hit ID shape.")
    
    # Convert MPPC IDs to positions
    mppc_positions = convert_mppc_to_location(
        mppc_id,
        mppc_col_id,
        mppc_positions,
        padding_value=padding_value,
    )
    if mppc_positions.shape[0] != mppc_id.shape[0]:
        raise ValueError("MPPC positions shape does not match MPPC ID shape.")
    
    # Adjust timestamps
    pixel_hit_timestamp = adjust_timestamps(pixel_hit_timestamp, padding_value)
    mppc_time = adjust_timestamps(mppc_time, padding_value)

    pixel_hit_spacetime = np.concatenate(
        [pixel_positions, pixel_hit_timestamp[:, :, None]], axis=-1
    )

    mppc_hit_spacetime = np.concatenate(
        [mppc_positions, mppc_time[:, :, None]], axis=-1
    )
    

    # Sort pixel hits by timestamp
    pixel_hit_spacetime = sort_by_feature(
        pixel_hit_spacetime, pixel_hit_timestamp, padding_value=padding_value
    )
    mppc_hit_spacetime = sort_by_feature(
        mppc_hit_spacetime, mppc_time, padding_value=padding_value
    )
    # Reorder to ensure non-padded entries are at the beginning
    pixel_hit_spacetime = reorder_nla(pixel_hit_spacetime, padding_value=padding_value)
    mppc_hit_spacetime = reorder_nla(mppc_hit_spacetime, padding_value=padding_value)

    pixel_hit_number = (pixel_hit_spacetime != padding_value).any(axis = -1).sum(axis=-1)
    mppc_hit_number = (mppc_hit_spacetime != padding_value).any(axis = -1).sum(axis=-1)

    valid_mask = (pixel_hit_number > 0) & (mppc_hit_number > 0)

    # Apply valid mask to spacetime data
    pixel_hit_spacetime = pixel_hit_spacetime[valid_mask]
    mppc_hit_spacetime = mppc_hit_spacetime[valid_mask]

    # Save spacetime data to files
    pixel_file_path = f"{out_dir}{out_name}_pixel_spacetime.npy"
    mppc_file_path = f"{out_dir}{out_name}_mppc_spacetime.npy"
    np.save(pixel_file_path, pixel_hit_spacetime)
    np.save(mppc_file_path, mppc_hit_spacetime)
    return pixel_hit_spacetime, mppc_hit_spacetime

In [ ]:
def convert_pixel_id_to_nla(
    pixel_id: np.ndarray, padding_value: int = -1
) -> np.ndarray:
    nla = np.full((*pixel_id.shape, 4), padding_value, dtype=np.int32)
    valid_mask = pixel_id != padding_value

    chip_id = pixel_id // 2**16
    station = chip_id // 2**12
    layer = ((chip_id // 2**10) % 4) + 1
    phi = ((chip_id // 2**5) % 2**5) + 1
    z_prime = chip_id % 2**5

    z = np.where(layer == 3, z_prime - 7, np.where(layer == 4, z_prime - 6, z_prime))

    station_mask = station == 0
    valid_mask = valid_mask & station_mask

    nla[valid_mask, 0] = station[valid_mask]
    nla[valid_mask, 1] = layer[valid_mask]
    nla[valid_mask, 2] = phi[valid_mask]
    nla[valid_mask, 3] = z[valid_mask]

    return nla

def get_rolled_up_sensor_hits(
    pixel_hit_ids: np.ndarray,
    padding_value: int = -1,
):
    """
    Converts pixel hit IDs to a rolled-up sensor hit representation.
    The whole detector is represented as a four 2D-images, where each image corresponds to a layer.
    """
    chip_ids = pixel_hit_ids // 2**16
    layer_ids = ((chip_ids // 2**10) % 4) + 1
    phi_ids = ((chip_ids // 2**5) % 2**5) + 1
    z_prime = chip_ids % 2**5
    z_ids = np.where(layer_ids == 3, z_prime - 7, np.where(layer_ids == 4, z_prime - 6, z_prime))
    station = chip_ids // 2**12
    number_of_events = pixel_hit_ids.shape[0]
    num_layers = 4

    valid_mask = pixel_hit_ids != padding_value
    valid_mask &= station == 0
    layer_images = []
    for layer in range(1, num_layers + 1):
        layer_mask = layer_ids == layer
        valid_layer_mask = valid_mask & layer_mask

        layer_phi_min = phi_ids[valid_layer_mask].min()
        layer_phi_max = phi_ids[valid_layer_mask].max()
        layer_z_min = z_ids[valid_layer_mask].min()
        layer_z_max = z_ids[valid_layer_mask].max()
        layer_image = np.zeros((number_of_events, np.unique(phi_ids[valid_layer_mask]).size, np.unique(z_ids[valid_layer_mask]).size, 1), dtype=np.int32)
        for event in range(number_of_events):
            for phi, z in zip(
                phi_ids[event][valid_layer_mask[event]],
                z_ids[event][valid_layer_mask[event]],
            ):
                if phi < layer_phi_min or phi > layer_phi_max or z < layer_z_min or z > layer_z_max:
                    raise ValueError(
                        f"Invalid phi or z value for event {event}: phi={phi}, z={z}, "
                        f"expected range: phi [{layer_phi_min}, {layer_phi_max}], z [{layer_z_min}, {layer_z_max}]"
                    )
                layer_image[event, phi - layer_phi_min, z - layer_z_min,0] +=1

        layer_images.append(layer_image)
    return layer_images


def get_timed_rolled_up_sensor_hits(
    pixel_hit_ids: np.ndarray,
    pixel_hit_timestamps: np.ndarray,
    padding_value: int = -1,
):
    """
    Converts pixel hit IDs to a rolled-up sensor hit representation.
    The whole detector is represented as a four 2D-images, where each image corresponds to a layer.
    """
    if pixel_hit_ids.shape != pixel_hit_timestamps.shape:
        raise ValueError("Pixel hit IDs and timestamps must have the same shape.")
    chip_ids = pixel_hit_ids // 2**16
    layer_ids = ((chip_ids // 2**10) % 4) + 1
    phi_ids = ((chip_ids // 2**5) % 2**5) + 1
    z_prime = chip_ids % 2**5
    z_ids = np.where(layer_ids == 3, z_prime - 7, np.where(layer_ids == 4, z_prime - 6, z_prime))
    station = chip_ids // 2**12
    number_of_events = pixel_hit_ids.shape[0]
    num_layers = 4

    valid_mask = pixel_hit_ids != padding_value
    valid_mask &= station == 0
    pixel_hit_timestamps[valid_mask] = -1
    pixel_hit_timestamps = adjust_timestamps(pixel_hit_timestamps, padding_value=padding_value)

    time_min = int(pixel_hit_timestamps[valid_mask].min())
    time_max = int(pixel_hit_timestamps[valid_mask].max())
    time_range = time_max - time_min + 1 

    layer_images = []
    for layer in range(1, num_layers + 1):
        layer_mask = layer_ids == layer
        valid_layer_mask = valid_mask & layer_mask

        layer_phi_min = phi_ids[valid_layer_mask].min()
        layer_phi_max = phi_ids[valid_layer_mask].max()
        layer_z_min = z_ids[valid_layer_mask].min()
        layer_z_max = z_ids[valid_layer_mask].max()
        layer_image = np.zeros((time_range, number_of_events, np.unique(phi_ids[valid_layer_mask]).size, np.unique(z_ids[valid_layer_mask]).size, 1), dtype=np.int32)
        for event in range(number_of_events):
            for phi, z, timestamp in zip(
                phi_ids[event][valid_layer_mask[event]],
                z_ids[event][valid_layer_mask[event]],
                pixel_hit_timestamps[event][valid_layer_mask[event]],
            ):
                if phi < layer_phi_min or phi > layer_phi_max or z < layer_z_min or z > layer_z_max:
                    raise ValueError(
                        f"Invalid phi or z value for event {event}: phi={phi}, z={z}, "
                        f"expected range: phi [{layer_phi_min}, {layer_phi_max}], z [{layer_z_min}, {layer_z_max}]"
                    )
                layer_image[timestamp - time_min, event, phi - layer_phi_min, z - layer_z_min,0] +=1

        layer_images.append(layer_image)
    return layer_images


In [ ]:
def get_image_slices_from_file(
    file_path: str, out_dir: str, out_name: str, padding_value: int = -1, hit_cutoff: int = 128
):
    with uproot.open(file_path) as file:
        event_data = file["mu3e"].arrays(library="pd")
        if event_data.empty:
            raise ValueError("Event data is empty.")
    pixel_hit_ids = load_ak_series_to_numpy(
        event_data["hit_pixelid"], max_cols=hit_cutoff, fill_value=padding_value
    )
    if pixel_hit_ids.shape[0] == 0:
        raise ValueError("No valid pixel hit IDs found.")
    images = get_rolled_up_sensor_hits(pixel_hit_ids, padding_value=padding_value)
    if not out_dir.endswith("/"):
        out_dir += "/"
    if not out_name:
        raise ValueError("Output name must be provided.")
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    with open(f"{out_dir}{out_name}_images.pkl", "wb") as f:
        pickle.dump(images, f)

def get_timed_image_slices_from_file(
    file_path: str, out_dir: str, out_name: str, padding_value: int = -1, hit_cutoff: int = 128
):
    with uproot.open(file_path) as file:
        event_data = file["mu3e"].arrays(library="pd")
        if event_data.empty:
            raise ValueError("Event data is empty.")
    pixel_hit_ids, pixel_hit_timestamps  = load_event_ak_to_numpy(
        [event_data["hit_pixelid"],
        event_data["hit_timestamp"]],
        cutoff=hit_cutoff,
        fill_value=padding_value,
    )    
    if pixel_hit_ids.shape[0] == 0:
        raise ValueError("No valid pixel hit IDs found.")
    images = get_timed_rolled_up_sensor_hits(pixel_hit_ids, pixel_hit_timestamps, padding_value=padding_value)
    if not out_dir.endswith("/"):
        out_dir += "/"
    if not out_name:
        raise ValueError("Output name must be provided.")
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    with open(f"{out_dir}{out_name}_images.pkl", "wb") as f:
        pickle.dump(images, f)

In [ ]:
DATA_DIR = "../mu3e_trigger_data"
ROOT_DATA_DIR = "../mu3e_root_data"
signal_data_file = f"{ROOT_DATA_DIR}/run42_sig.root"
background_data_file = f"{ROOT_DATA_DIR}/run42_bg.root"
e5_data_file = f"{ROOT_DATA_DIR}/run42_5e-sort.root"
familong_data_file = f"{ROOT_DATA_DIR}/run42_familon-sort.root"
signal_only_data_file = f"{ROOT_DATA_DIR}/run42_sig_only-sort.root"
HIT_COUNT_CUTOFF = 128


In [ ]:
get_image_slices_from_file(
    file_path=signal_data_file,
    out_dir=DATA_DIR,
    out_name="sig",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_image_slices_from_file(
    file_path=background_data_file,
    out_dir=DATA_DIR,
    out_name="bg",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_image_slices_from_file(
    file_path=e5_data_file,
    out_dir=DATA_DIR,
    out_name="e5",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_image_slices_from_file(
    file_path=familong_data_file,
    out_dir=DATA_DIR,
    out_name="familong",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_image_slices_from_file(
    file_path=signal_only_data_file,
    out_dir=DATA_DIR,
    out_name="sig_only",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)

In [ ]:
get_timed_image_slices_from_file(
    file_path=signal_data_file,
    out_dir=DATA_DIR,
    out_name="sig",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_timed_image_slices_from_file(
    file_path=background_data_file,
    out_dir=DATA_DIR,
    out_name="bg",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_timed_image_slices_from_file(
    file_path=e5_data_file,
    out_dir=DATA_DIR,
    out_name="e5",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_timed_image_slices_from_file(
    file_path=familong_data_file,
    out_dir=DATA_DIR,
    out_name="familong",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_timed_image_slices_from_file(
    file_path=signal_only_data_file,
    out_dir=DATA_DIR,
    out_name="sig_only",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)

In [ ]:

get_spacetime_from_file(
    file_path=signal_data_file,
    out_dir=DATA_DIR,
    out_name="sig",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_spacetime_from_file(
    file_path=background_data_file,
    out_dir=DATA_DIR,
    out_name="bg",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)

get_spacetime_from_file(
    file_path=e5_data_file,
    out_dir=DATA_DIR,
    out_name="5e",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_spacetime_from_file(
    file_path=familong_data_file,
    out_dir=DATA_DIR,
    out_name="familon",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)
get_spacetime_from_file(
    file_path=signal_only_data_file,
    out_dir=DATA_DIR,
    out_name="sig_only",
    padding_value=-1,
    hit_cutoff=HIT_COUNT_CUTOFF,
)


In [ ]:
def convert_pixel_id_to_nla(
    pixel_id: np.ndarray, padding_value: int = -1
) -> np.ndarray:
    nla = np.full((*pixel_id.shape, 4), padding_value, dtype=np.int32)
    valid_mask = pixel_id != padding_value

    chip_id = pixel_id // 2**16
    station = chip_id // 2**12
    layer = ((chip_id // 2**10) % 4) + 1
    phi = ((chip_id // 2**5) % 2**5) + 1
    z_prime = chip_id % 2**5

    z = np.where(layer == 3, z_prime - 7, np.where(layer == 4, z_prime - 6, z_prime))

    station_mask = station == 0
    valid_mask = valid_mask & station_mask

    nla[valid_mask, 0] = station[valid_mask]
    nla[valid_mask, 1] = layer[valid_mask]
    nla[valid_mask, 2] = phi[valid_mask]
    nla[valid_mask, 3] = z[valid_mask]

    return nla